# Hugging Face: use an existing model as a tool
**Faculty AI workshop · CloudBank Classroom · CPU · optional third-part demo (15 minutes)**

Run a small pretrained sentiment classifier, then call a public Gradio app as a remote
tool. This connects the short-text activity to reusable models and APIs.

**Questions:** Does a sentiment model implement our extraction rubric? What happens
to neutral or mixed text when the model has only two labels? Where does computation happen?

The prepared model uses approximately 268 MB of weights. Pre-download it before class
and allow additional memory for PyTorch. A CPU is sufficient; use the three scientific
notebooks if this environment cannot fit or download it. Public Hub model downloads
need no API token. These demos do not invoke paid inference providers.


## 1. Instructor dependency check
Use a Python 3.10+ kernel. Install dependencies before class if your classroom permits it,
then restart the kernel. The scientific notebooks do not need these extra dependencies.


In [ ]:
import sys, importlib.util
packages = ["torch", "transformers", "pandas", "huggingface_hub", "gradio_client", "ipywidgets"]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
print("Python:", sys.version.split()[0], "| Missing packages:", missing or "none")
# Optional instructor setup (uncomment if needed):
# %pip install 'transformers>=4.45,<5' torch pandas huggingface-hub gradio-client
if missing:
    raise ImportError("Prepare the Hugging Face dependencies before running this demo.")


## 2. Read the model card before running it
We use `distilbert/distilbert-base-uncased-finetuned-sst-2-english` (Apache 2.0).
It was fine-tuned on English movie-review sentiment (SST-2). It has **POSITIVE** and
**NEGATIVE** labels; it cannot return neutral or mixed, extract adverse effects, or
judge the truth of a health statement. A high softmax score is not a calibrated
probability of correctness on our workshop data.

[Model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english)
· [Transformers pipelines documentation](https://huggingface.co/docs/transformers/main_classes/pipelines)


In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display
roots = [Path.cwd(), Path.cwd().parent, Path.cwd() / "workshop"]
ROOT = next((p for p in roots if (p / "notebooks").is_dir()), Path.cwd())
# Save the cache inside the workshop folder so your classroom copy can reuse it.
os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
REVISION = "714eb0fa89d2f80546fda750413ed43d93601a13"  # verified Hub commit
from transformers import pipeline
sentiment = pipeline("sentiment-analysis", model=MODEL_ID, revision=REVISION,
                     device=-1, trust_remote_code=False, model_kwargs={"use_safetensors": True})
print("Loaded", MODEL_ID, "revision", REVISION, "on CPU")


## 3. Run a model; inspect its output
The six sentences below are **instructor-authored teaching examples**, not dataset
records. Human labels follow the four-category workshop rubric and include mixed and
neutral examples to expose the model's two-label constraint.


In [ ]:
examples = pd.DataFrame([
    {"text": "The food was delicious and the staff were wonderful.", "human_rubric": "positive"},
    {"text": "The product broke immediately and the support was terrible.", "human_rubric": "negative"},
    {"text": "The package arrived on Tuesday.", "human_rubric": "neutral"},
    {"text": "The screen is excellent, but the battery is terrible.", "human_rubric": "mixed"},
    {"text": "I expected to hate this film, but I loved it.", "human_rubric": "positive"},
    {"text": "The story was neither good nor bad.", "human_rubric": "neutral"},
])
predictions = sentiment(examples.text.tolist(), truncation=True, max_length=512, batch_size=4)
examples["model_label"] = [p["label"].lower() for p in predictions]
examples["model_score"] = [p["score"] for p in predictions]
examples["rubric_has_label_model_cannot_return"] = examples.human_rubric.isin(["neutral", "mixed"])
display(examples)
print("Discuss disagreements; six selected examples are not a model accuracy benchmark.")


## 4. Wrap it in an analysis tool
A tool is a callable function with inputs and outputs. The same function can be
reused from a notebook, connected to an app, or offered to an AI agent. Here a
text box is sufficient; no agent orchestration is needed to demonstrate the idea.


In [ ]:
def classify_review(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Enter a nonempty review.")
    pred = sentiment(text, truncation=True, max_length=512)[0]
    return {"text": text, "label": pred["label"].lower(), "score": round(pred["score"], 4),
            "model": MODEL_ID, "revision": REVISION,
            "note": "Binary sentiment only; inputs longer than 512 tokens are truncated."}

display(classify_review("The seminar was useful, but the room was uncomfortable."))


In [ ]:
# The notebook's text-box tool. The function call above works if widgets are unavailable.
import ipywidgets as widgets
text_box = widgets.Textarea(value="The seminar was useful, but the room was uncomfortable.",
                           description="Review:", layout=widgets.Layout(width="85%"))
button = widgets.Button(description="Classify locally", button_style="info")
output = widgets.Output()
def run_review(_):
    with output:
        output.clear_output(wait=True)
        display(classify_review(text_box.value))
button.on_click(run_review)
display(text_box, button, output)


## 5. Compare with the real reviews from Part 1 (optional)
Use the bundled UCI review sentences as a bridge to the earlier prompt activity.
Their original annotation is binary sentiment, so it is compatible with this model's
label set. A workshop convenience sample still does not establish general performance.
Inspect the actual JSONL file and choose the text/label fields in the next cell.


In [ ]:
import json
review_path = ROOT / "part1/data/reviews.jsonl"
if review_path.exists():
    records = [json.loads(line) for line in review_path.read_text().splitlines() if line.strip()]
    review_texts = [r["text"] for r in records[:10]]
    results = sentiment(review_texts, truncation=True, max_length=512, batch_size=4)
    display(pd.DataFrame({"text": review_texts,
                          "model_label": [r["label"] for r in results],
                          "model_score": [r["score"] for r in results]}))
else:
    print("Optional: distribute the full workshop folder to use the Part 1 review sample.")


## 6. Call an existing tool hosted on Hugging Face Spaces
**Models** contain downloadable trained weights. **Spaces** host apps; a Gradio app
can expose callable endpoints. The call below performs computation on a remote Space.
We send only the literal numbers 4 and 5 to the official Gradio calculator example.
Review the input policy before sending your own course data to a remote service.

Set `RUN_REMOTE_SPACE = True` to connect. Public Spaces can sleep, change their API,
queue requests, or be unavailable; the scientific notebooks remain usable offline.
`view_api()` shows the current signature rather than guessing argument names.

[Space](https://huggingface.co/spaces/gradio/calculator)
· [Official Python client guide](https://gradio.app/guides/getting-started-with-the-python-client)


In [ ]:
RUN_REMOTE_SPACE = False  # instructor opt-in: the next cell sends a remote request
if RUN_REMOTE_SPACE:
    from gradio_client import Client
    client = Client("gradio/calculator", httpx_kwargs={"timeout": 30.0})
    client.view_api()
else:
    print("Remote Space call off. Set RUN_REMOTE_SPACE=True to inspect its API.")


In [ ]:
if RUN_REMOTE_SPACE:
    # Signature documented by Gradio; inspect view_api above in case the Space changed.
    job = client.submit(num1=4, operation="add", num2=5, api_name="/predict")
    try:
        print("Remote calculator result:", job.result(timeout=30))
    except TimeoutError:
        job.cancel()
        print("The public Space did not respond within 30 seconds; use the local model demo.")


## 7. Teaching prompts and next steps
> Explain `pipeline` and `classify_review` for a faculty member who has never used
> Python. Identify what is downloaded, where the prediction runs, what is sent to a
> remote service, and which rubric categories the model cannot return.

> Using the existing classifier, create a table that groups review predictions by
> label and counts them. Preserve the source text and model revision. Do not describe
> the score as accuracy, and do not convert the binary model into a four-category classifier.

**Extension:** Choose a domain-specific model or Space from Hugging Face. Inspect
its card, license, task, input format, output labels, hardware needs, and evaluation
domain. For microscopy, a segmentation model must be compared with reference masks
from an appropriate modality. A general image classifier is not a cell-counting tool.
For ocean profiles, deterministic analysis often answers the teaching question directly;
an LLM can help implement and explain the tool without replacing the measurement.

**Exit ticket:** Name one useful AI interaction, one validation you would require,
and one change you would make before using this exercise in your course.
